# Phase 2.2: Model Optimizasyonu ve Kuantizasyon (Quantization)
Bu notebook, Streamlit web uygulamasına entegre edeceğimiz modelleri **hızlandırmak** ve **boyutlarını küçültmek** için hazırlanmıştır (Capstone Task 2.1).

Hedef:
- Modellerin inference (tahmin) süresini **< 2 saniye** altına düşürmek.
- CPU üzerinde çalışan sunucularda (Streamlit Cloud) daha az RAM tüketmesi için Modeli `float32`'den `int8` formatına sıkıştırmak (Post-Training Dynamic Quantization).

In [2]:
import os
import time
import torch
import timm

# CPU üzerinden test edeceğiz çünkü Streamlit Cloud genelde CPU sunucularında çalışır.
DEVICE = torch.device('cpu')
print(f"Test Ortamı: {DEVICE}")

Test Ortamı: cpu


## 1. Orijinal Modellerin Yüklenmesi ve İncelenmesi

In [3]:
num_classes = 102

# 1. ViT Modelini Yükle
vit_model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=num_classes)
vit_model.load_state_dict(torch.load('best_vit_model.pth', map_location=DEVICE, weights_only=True))
vit_model.eval()

# 2. EfficientNet Modelini Yükle
eff_model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=num_classes)
eff_model.load_state_dict(torch.load('best_efficientnet_model.pth', map_location=DEVICE, weights_only=True))
eff_model.eval()

def print_model_size(file_path, model_name):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"{model_name} Dosya Boyutu: {size_mb:.2f} MB")

print_model_size('best_vit_model.pth', 'Orijinal ViT')
print_model_size('best_efficientnet_model.pth', 'Orijinal EfficientNet')

Orijinal ViT Dosya Boyutu: 327.65 MB
Orijinal EfficientNet Dosya Boyutu: 68.37 MB


## 2. Gecikme (Latency) Testi Fonksiyonu

In [4]:
def measure_inference_latency(model, num_samples=30):
    # Rastgele bir resim tensörü oluşturalım (1 resim, 3 kanal, 224x224)
    dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    
    # Isınma turları (Modelin bellek tahsisi yapması için)
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_input)
            
    # Gerçek ölçüm
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_samples):
            _ = model(dummy_input)
    end_time = time.time()
    
    avg_latency = (end_time - start_time) / num_samples
    print(f"Ortalama Inference Süresi: {avg_latency:.4f} saniye (Hedef: < 2.0 s)")
    return avg_latency

print("--- Orijinal Modellerin Hızları ---")
print("ViT:")
vit_latency_orig = measure_inference_latency(vit_model)
print("EfficientNet:")
eff_latency_orig = measure_inference_latency(eff_model)

--- Orijinal Modellerin Hızları ---
ViT:
Ortalama Inference Süresi: 0.2160 saniye (Hedef: < 2.0 s)
EfficientNet:
Ortalama Inference Süresi: 0.0822 saniye (Hedef: < 2.0 s)


## 3. Dinamik Kuantizasyon (Dynamic Quantization) Uygulanması
PyTorch'un `quantize_dynamic` aracı, modelin içindeki `nn.Linear` (Dense/Fully Connected) katmanlarının ağırlıklarını 32-bit Float'tan 8-bit Integer formatına dönüştürür. Bu, ViT gibi içinde çok fazla Linear katman bulunduran Transformer modellerinde **devasa** bir sıkıştırma sağlar!

In [5]:
# ViT Kuantizasyonu (Transformer modellerinde Linear katmanlar çoğunluktadır, bu yüzden harika sonuç verir)
print("ViT modeli kuantize ediliyor...")
quantized_vit = torch.ao.quantization.quantize_dynamic(
    vit_model, {torch.nn.Linear}, dtype=torch.qint8
)

# EfficientNet Kuantizasyonu (CNN modellerinde Linear katman azdır (sadece sondaki sınıflandırıcı), ancak yine de uygulanabilir)
print("EfficientNet modeli kuantize ediliyor...")
quantized_eff = torch.ao.quantization.quantize_dynamic(
    eff_model, {torch.nn.Linear}, dtype=torch.qint8
)

print("Kuantizasyon tamamlandı!")

ViT modeli kuantize ediliyor...
EfficientNet modeli kuantize ediliyor...
Kuantizasyon tamamlandı!


## 4. Kuantize Edilmiş Modellerin Kaydedilmesi ve İncelenmesi

In [6]:
# Yeni modelleri kaydedelim
torch.save(quantized_vit.state_dict(), 'quantized_vit_model.pth')
torch.save(quantized_eff.state_dict(), 'quantized_efficientnet_model.pth')

# Yeni Boyutları Yazdıralım
print("--- Kuantize Edilmiş Modellerin Boyutları ---")
print_model_size('quantized_vit_model.pth', 'Quantized ViT')
print_model_size('quantized_efficientnet_model.pth', 'Quantized EfficientNet')

# Yeni Hızları Ölçelim
print("\n--- Kuantize Edilmiş Modellerin Hızları ---")
print("Quantized ViT:")
vit_latency_quant = measure_inference_latency(quantized_vit)
print("Quantized EfficientNet:")
eff_latency_quant = measure_inference_latency(quantized_eff)

--- Kuantize Edilmiş Modellerin Boyutları ---
Quantized ViT Dosya Boyutu: 84.46 MB
Quantized EfficientNet Dosya Boyutu: 67.84 MB

--- Kuantize Edilmiş Modellerin Hızları ---
Quantized ViT:
Ortalama Inference Süresi: 0.1278 saniye (Hedef: < 2.0 s)
Quantized EfficientNet:
Ortalama Inference Süresi: 0.0962 saniye (Hedef: < 2.0 s)


## 5. Sonuç Özeti
Oluşan `quantized_vit_model.pth` veya `quantized_efficientnet_model.pth` dosyasını Streamlit web sitenizde kullanacaksınız.